# Tutorial 05: Negotiation Pattern in Multi-Agent Systems (Offline-First)

Focused tutorial for `A2A/05_Negotiation`, where multiple agents are queried, responses are compared, and one result is selected.

## 1) Where We Are in the Journey

`04_multi_agent_workflow` introduced ordered multi-step execution.
This step answers a different question: when multiple agents can respond, how do we evaluate and choose?
Negotiation adds post-execution decision intelligence.

## 2) What You Will Learn

- Broadcast-style negotiation flow
- Tool-specific payload building per candidate agent
- Response collection and evaluation logic
- Why selection-after-execution can outperform pre-routing for ambiguous tasks

## 3) Why This Matters

Some queries are ambiguous and multiple agents may appear plausible.
Negotiation allows evidence-driven selection from actual responses.
This improves robustness when confidence before execution is low.

## 4) Core Concept Deep Dive

Intuition: ask multiple agents, then decide.
Architecture in this folder: `discover_capabilities` -> `negotiate` (broadcast) -> `evaluate` (pick success).
Trade-off: more reliability and flexibility vs higher latency/cost.
Design detail: `build_payload` is tool-aware and agent-specific.

## 5) Code Walkthrough (Only `A2A/05_Negotiation`)

- `discover_capabilities()` builds agent registry.
- `build_payload(query, agent)` tries to map query to one tool in each agent.
- `negotiate(query, registry)` broadcasts and collects responses.
- `evaluate(responses)` selects first successful result.
- `execute(query)` orchestrates full negotiation loop.

In [ ]:
import re

MOCK_REGISTRY = [
    {'agent': 'math-agent', 'endpoint': 'http://localhost:8000/invoke', 'tools': [{'name': 'add', 'description': 'Add two numbers'}]},
    {'agent': 'finance-agent', 'endpoint': 'http://localhost:8001/invoke', 'tools': [{'name': 'calculate_interest', 'description': 'Calculate simple interest'}]}
]

def discover_capabilities():
    return MOCK_REGISTRY

In [ ]:
def build_payload(query, agent):
    query = query.lower()
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
    for tool in agent['tools']:
        name = tool['name']
        if name == 'add' and 'add' in query and len(numbers) >= 2:
            return {'tool': 'add', 'args': {'a': numbers[0], 'b': numbers[1]}}
        if name == 'calculate_interest' and 'interest' in query and len(numbers) >= 3:
            return {'tool': 'calculate_interest', 'args': {'principal': numbers[0], 'rate': numbers[1], 'time': numbers[2]}}
    return None

def invoke_sim(agent, payload):
    if payload is None:
        return {'status': 'error', 'message': 'No matching tool'}
    if agent['agent'] == 'math-agent' and payload['tool'] == 'add':
        a = payload['args']
        return {'status': 'success', 'result': a['a'] + a['b']}
    if agent['agent'] == 'finance-agent' and payload['tool'] == 'calculate_interest':
        a = payload['args']
        return {'status': 'success', 'result': (a['principal'] * a['rate'] * a['time']) / 100}
    return {'status': 'error', 'message': 'Unsupported'}

In [ ]:
def negotiate(query, registry):
    responses = []
    for agent in registry:
        payload = build_payload(query, agent)
        res = invoke_sim(agent, payload)
        responses.append({'agent': agent['agent'], 'payload': payload, 'response': res})
    return responses

def evaluate(responses):
    for r in responses:
        if r['response'].get('status') == 'success':
            return r
    return {'error': 'No valid response'}

def execute(query):
    registry = discover_capabilities()
    responses = negotiate(query, registry)
    best = evaluate(responses)
    return responses, best

In [ ]:
responses, best = execute('add 10 and 20')
print('RESPONSES:')
for r in responses:
    print(r)
print('SELECTED:', best)

## 6) System Flow Explanation

1. Discover candidate agents.
2. Build candidate payload per agent.
3. Execute each candidate call.
4. Collect response set.
5. Evaluate and pick best valid response.

## 7) Important Concepts You Might Miss

- Negotiation shifts decision point after execution.
- Broadcast increases cost but improves selection confidence.
- Evaluation policy is a strategic component, not utility glue.

## 8) Common Mistakes / Pitfalls

- Over-broadcasting when one agent is sufficient.
- Weak evaluate rules that accept low-quality outputs.
- Payload mismatch causing false negatives for capable agents.

## 9) Key Takeaways

- Negotiation is execute-many then decide-one.
- It complements routing when ambiguity is high.
- Evaluation quality determines system reliability.

## 10) What’s Next (Strict Preview)

`06_Multi-Agent_Workflow` formalizes workflow control structures (state, routing, invocation, validation) into a clearer orchestration loop.
It solves how to make multi-agent execution extensible rather than ad-hoc.
This matters as workflows grow in complexity and failure modes.